# Classification du Type de Documents Medicaux — Modele a Risque Minimal (AI Act)

**Risque minimal** — Outil de tri documentaire sans impact sur les soins

## 0. Dependances optionnelles

In [ ]:
# Author: Octo Technology MLOps Tribe
%pip install shap pandera --quiet

## 1. Imports

In [ ]:
# Author: Octo Technology MLOps Tribe
import json
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pandera as pa
from mlflow.models.signature import infer_signature
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import issparse

warnings.filterwarnings("ignore")
ARTIFACTS_DIR = Path(tempfile.mkdtemp())
print(f"Repertoire artefacts temporaires : {ARTIFACTS_DIR}")

## 2. Configuration MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
PROJECT_NAME = "Medical-Document-NLP"
MLFLOW_TRACKING_URI = f"http://model-platform.com/registry/{PROJECT_NAME}/"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("medical_document_classification")
print(f"MLflow tracking URI : {MLFLOW_TRACKING_URI}")

## 3. Generation des donnees synthetiques

In [ ]:
# Author: Octo Technology MLOps Tribe
np.random.seed(42)
random.seed(42)
N = 3000

import random

DOC_TYPES = {
    0: {
        "name": "compte-rendu de consultation",
        "templates": [
            "Compte-rendu de consultation du {date}. Patient suivi pour {motif}. Examen clinique : {exam}. Conclusion : {concl}.",
            "Consultation specialisee. Motif : {motif}. Antecedents : {atcd}. Plan therapeutique : {plan}.",
        ],
    },
    1: {
        "name": "ordonnance",
        "templates": [
            "Ordonnance medicale. Prescrit : {medicament} {posologie} pendant {duree}. Renouvellement : {renouvellement}.",
            "Je soussigne prescris : {medicament} {posologie}. {duree}. Ne pas depasser la dose prescrite.",
        ],
    },
    2: {
        "name": "compte-rendu d'hospitalisation",
        "templates": [
            "Compte-rendu d'hospitalisation du {date_entree} au {date_sortie}. Motif d'admission : {motif}. Evolution : {evolution}. Sortie : {sortie}.",
            "Hospitalisation pour {motif}. Bilan realise : {bilan}. Traitement installe : {traitement}. Suivi ambulatoire recommande.",
        ],
    },
    3: {
        "name": "resultat d'examen",
        "templates": [
            "Resultat d'analyse : {examen}. Valeur : {valeur} {unite}. Norme : {norme}. Interpretation : {interpretation}.",
            "Compte-rendu d'imagerie. Type : {examen}. Indication : {indication}. Resultat : {resultat}.",
        ],
    },
    4: {
        "name": "lettre de sortie",
        "templates": [
            "Lettre de sortie du {date}. Cher confrere, je vous adresse ce patient pour {raison}. Diagnostic retenu : {diagnostic}. Traitement de sortie : {traitement}.",
            "Courrier de liaison. Suite a l'hospitalisation pour {raison}, le patient sort ce jour avec : {traitement}. Prochain RDV : {rdv}.",
        ],
    },
}

FILLERS = {
    "date": ["12/01/2024", "03/03/2024", "15/06/2024"],
    "motif": ["douleur abdominale", "bilan cardiologique", "suivi diabete", "lombalgie aigue"],
    "exam": ["normal", "sans anomalie", "discret souffle systolique", "abdomen souple"],
    "concl": ["RAS", "surveillance", "traitement adapte", "bilan complementaire demande"],
    "atcd": ["HTA connue", "diabete type 2", "allergie penicilline", "sans antecedent notable"],
    "plan": ["poursuite traitement", "ajustement posologie", "kinesitherapie", "consultation cardiologie"],
    "medicament": ["metformine 1g", "amlodipine 5mg", "omeprazole 20mg", "sertraline 50mg"],
    "posologie": ["1 cp matin", "2 cp/jour", "1 cp au coucher", "3 cp/jour pendant les repas"],
    "duree": ["30 jours", "3 mois", "6 mois", "traitement continu"],
    "renouvellement": ["non renouvelable", "renouvelable 2 fois", "renouvelable sur avis medical"],
    "date_entree": ["10/01/2024"], "date_sortie": ["14/01/2024"],
    "evolution": ["favorable", "stable", "amelioration progressive", "complication geree"],
    "sortie": ["retour domicile", "transfert SSR", "sortie avec HAD"],
    "bilan": ["ECG normal", "biologie standard", "scanner thoracique", "echo cardiaque"],
    "traitement": ["metformine + amlodipine", "antalgiques + AINS", "antibiotiques 7j"],
    "examen": ["glycemie", "NFS", "echo abdominale", "radio thorax", "IRM lombaire"],
    "valeur": ["6.8", "12.5", "140", "98", "0.5"],
    "unite": ["mmol/L", "g/dL", "mmHg", "U/L", "mg/L"],
    "norme": ["[4-6 mmol/L]", "[12-16 g/dL]", "[<14 mmHg]"],
    "interpretation": ["normal", "dans les normes", "legerement eleve — a surveiller", "pathologique"],
    "indication": ["douleur", "suivi", "bilan"],
    "resultat": ["sans anomalie", "hernie discale L4-L5", "epanchement pleural minime"],
    "raison": ["bilan cardiologique", "prise en charge diabete", "episode infectieux"],
    "diagnostic": ["insuffisance cardiaque stade II", "diabete desequilibre", "pneumopathie"],
    "rdv": ["dans 1 mois", "dans 3 mois avec le medecin traitant"],
}

texts, labels = [], []
for _ in range(N):
    label = random.randint(0, 4)
    doc_info = DOC_TYPES[label]
    tpl = random.choice(doc_info["templates"])
    filled = tpl
    for key, vals in FILLERS.items():
        placeholder = "{" + key + "}"
        if placeholder in filled:
            filled = filled.replace(placeholder, random.choice(vals), 1)
    texts.append(filled)
    labels.append(label)

TARGET = "doc_type"
df = pd.DataFrame({
    "text":       texts,
    "text_length": [len(t) for t in texts],
    "word_count":  [len(t.split()) for t in texts],
    TARGET:        labels,
})

PROTECTED_ATTRIBUTES = []
print(f"Dataset : {len(df):,} lignes | Taux cible : {df[TARGET].mean():.1%}")

## 4. Validation Pandera & statistiques descriptives

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA = pa.DataFrameSchema(
    name="medical_document_input_schema",
    description="Contrat de donnees — Classification du Type de Documents Medicaux — Modele a Risque Minimal (AI Act)",
    columns={
        "text":        pa.Column(str,  checks=pa.Check(lambda s: s.str.len() >= 10, error="Texte trop court"), nullable=False, description="Texte du document"),
        "text_length": pa.Column(int,  checks=pa.Check.in_range(10, 5000),   nullable=False, description="Longueur du texte (caracteres)"),
        "word_count":  pa.Column(int,  checks=pa.Check.in_range(2, 1000),    nullable=False, description="Nombre de mots"),
        TARGET:        pa.Column(int,  checks=pa.Check.isin(list(range(5))), nullable=False, description="Variable cible"),
    },
    coerce=False,
    strict=True,
)

try:
    SCHEMA.validate(df[["text", "text_length", "word_count", TARGET]], lazy=True)
    PANDERA_STATUS = "PASS"
    PANDERA_ERRORS = 0
    print("Validation Pandera : SUCCES")
except pa.errors.SchemaErrors as exc:
    PANDERA_STATUS = "FAIL"
    PANDERA_ERRORS = len(exc.failure_cases)
    print(f"Validation Pandera : ECHEC ({PANDERA_ERRORS} erreurs)")

In [ ]:
# Author: Octo Technology MLOps Tribe
print("=== Statistiques descriptives ===")
print(f"Nombre de documents : {len(df):,}")
print(f"Longueur moyenne des textes : {df['text_length'].mean():.0f} caracteres")
print(f"Nombre de mots moyen : {df['word_count'].mean():.0f}")
print(f"\n=== Distribution de la variable cible ===")
vc = df[TARGET].value_counts().sort_index()
for k, v in vc.items():
    print(f"  Classe {k} : {v:,} ({v/len(df):.1%})")

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA_YAML_EXPORTED = False
try:
    schema_yaml_path = ARTIFACTS_DIR / "pandera_schema.yaml"
    with open(schema_yaml_path, "w") as f:
        f.write(SCHEMA.to_yaml())
    SCHEMA_YAML_EXPORTED = True
except Exception as e:
    print(f"Export YAML non disponible : {e}")

validation_report = {
    "schema_name": SCHEMA.name,
    "validation_status": PANDERA_STATUS,
    "validation_errors": PANDERA_ERRORS,
    "n_rows_validated": int(len(df)),
    "protected_attributes": PROTECTED_ATTRIBUTES,
}
validation_report_path = ARTIFACTS_DIR / "data_validation_report.json"
with open(validation_report_path, "w") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print("Rapport de validation exporte.")

## 5. Pretraitement

In [ ]:
# Author: Octo Technology MLOps Tribe
X_text, y = df["text"], df[TARGET]

X_train_full, X_test, y_train_full, y_test = train_test_split(X_text, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full)

vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), sublinear_tf=True)
X_train_sc = vectorizer.fit_transform(X_train)
X_val_sc   = vectorizer.transform(X_val)
X_test_sc  = vectorizer.transform(X_test)

FEATURES = vectorizer.get_feature_names_out().tolist()
print(f"Train : {len(X_train):,}  |  Validation : {len(X_val):,}  |  Test : {len(X_test):,}")
print(f"Vocabulaire TF-IDF : {len(FEATURES):,} tokens")

## 6. Entrainement du modele

In [ ]:
# Author: Octo Technology MLOps Tribe
PARAMS = {"estimator": LinearSVC(C=1.0, max_iter=2000, random_state=42), "cv": 3}

model = CalibratedClassifierCV(**PARAMS)
model.fit(X_train_sc, y_train)

val_pred = model.predict(X_val_sc)
val_proba = model.predict_proba(X_val_sc)
if 5 == 2:
    val_auc = roc_auc_score(y_val, val_proba[:, 1])
else:
    val_auc = roc_auc_score(y_val, val_proba, multi_class="ovr", average="macro")
print(f"AUC-ROC validation : {val_auc:.4f}")
print("Entrainement termine.")

## 7. Evaluation sur le jeu de test

In [ ]:
# Author: Octo Technology MLOps Tribe
y_pred  = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)

if 5 == 2:
    auc = round(float(roc_auc_score(y_test, y_proba[:, 1])), 4)
else:
    auc = round(float(roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")), 4)

METRICS = {
    "accuracy":  round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred, average="macro", zero_division=0)), 4),
    "recall":    round(float(recall_score(y_test, y_pred, average="macro", zero_division=0)), 4),
    "f1_score":  round(float(f1_score(y_test, y_pred, average="macro", zero_division=0)), 4),
    "auc_roc":   auc,
}

report_dict = classification_report(y_test, y_pred, output_dict=True)
print(classification_report(y_test, y_pred))
print("\n".join(f"{k:<12}: {v}" for k, v in METRICS.items()))

In [ ]:
# Author: Octo Technology MLOps Tribe
# Multi-class: confusion matrix instead of ROC
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["CR Consultation", "Ordonnance", "CR Hospit.", "Resultat Exam.", "Lettre Sortie"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matrice de Confusion — Classification Documentaire")
plt.tight_layout()
roc_path = ARTIFACTS_DIR / "confusion_matrix.png"
plt.savefig(roc_path, dpi=150)
plt.show()

## 8. Analyse d'equite (Fairness)

In [ ]:
# Author: Octo Technology MLOps Tribe
# Modele NLP — pas d'attribut protege dans les features textuelles synthetiques
FAIRNESS_REPORT = {
    "protected_attributes": [],
    "note": "Aucun attribut protege dans les features TF-IDF synthetiques. En production, des biais linguistiques (genre, origine) pourraient etre presents dans les textes reels.",
    "global_accuracy": METRICS["accuracy"],
    "global_f1": METRICS["f1_score"],
}


fairness_path = ARTIFACTS_DIR / "fairness_report.json"
with open(fairness_path, "w") as f:
    json.dump(FAIRNESS_REPORT, f, indent=2, ensure_ascii=False)
print("Rapport d'equite exporte.")

## 9. Explicabilite

In [ ]:
# Author: Octo Technology MLOps Tribe
# Top features TF-IDF par importance (coefficients du modele)
if hasattr(model, "coef_"):
    coef = model.coef_
    if coef.ndim > 1:
        importances = np.abs(coef).mean(axis=0)
    else:
        importances = np.abs(coef[0])
    top_n = min(20, len(importances))
    top_idx = np.argsort(importances)[-top_n:][::-1]
    fi_df = pd.DataFrame({"feature": [FEATURES[i] for i in top_idx], "importance": importances[top_idx]})
else:
    fi_df = pd.DataFrame({"feature": FEATURES[:20], "importance": np.ones(20)})

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color="mediumorchid")
ax.set_title("Top tokens TF-IDF — Importance des features")
ax.set_xlabel("Importance relative (|coefficient|)")
plt.tight_layout()
fi_path = ARTIFACTS_DIR / "feature_importance.png"
plt.savefig(fi_path, dpi=150)
plt.show()
print(fi_df.head(10).to_string(index=False))

In [ ]:
# Author: Octo Technology MLOps Tribe
# SHAP non applicable directement aux modeles TF-IDF lineaires de cette maniere — on utilise LIME si disponible
SHAP_AVAILABLE = False
try:
    import lime
    import lime.lime_text
    explainer = lime.lime_text.LimeTextExplainer(class_names=[str(i) for i in range(model.classes_.max() + 1)])
    sample_text = X_test.iloc[0]
    exp = explainer.explain_instance(sample_text, lambda x: model.predict_proba(vectorizer.transform(x)), num_features=10)
    print("LIME explanation pour le premier exemple de test :")
    for feat, weight in exp.as_list():
        print(f"  {feat}: {weight:.4f}")
    SHAP_AVAILABLE = True
except ImportError:
    print("LIME non disponible. Installer avec : pip install lime")

## 10. Preparation des artefacts de documentation

In [ ]:
# Author: Octo Technology MLOps Tribe
cr_path = ARTIFACTS_DIR / "classification_report.json"
with open(cr_path, "w") as f:
    json.dump(report_dict, f, indent=2, ensure_ascii=False)

fi_csv_path = ARTIFACTS_DIR / "feature_importance.csv"
fi_df.to_csv(fi_csv_path, index=False)

pp_context = {"schema_name": SCHEMA.name, "pandera_status": PANDERA_STATUS, "pandera_errors": PANDERA_ERRORS}
preprocessing_md = Path("preprocessing_description_doctype_template.md").read_text(encoding="utf-8").format_map(pp_context)
pp_path = ARTIFACTS_DIR / "preprocessing_description.md"
pp_path.write_text(preprocessing_md, encoding="utf-8")

print("Artefacts prepares :")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"  {p.name}")

## 11. Logging MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
MODEL_NAME  = "document_type_classifier"
TEAM        = "mlops-tribe"
ENVIRONMENT = "staging"

model_card_text = Path("model_card_doctype.md").read_text(encoding="utf-8")
model_card_path = ARTIFACTS_DIR / "model_card.md"
model_card_path.write_text(model_card_text, encoding="utf-8")

with mlflow.start_run(run_name=f"{MODEL_NAME}_v1") as run:
    mlflow.log_params(PARAMS)
    mlflow.log_param("vectorizer",       "TfidfVectorizer")
    mlflow.log_param("feature_count",    len(FEATURES))
    mlflow.log_param("train_size",       len(X_train))
    mlflow.log_param("val_size",         len(X_val))
    mlflow.log_param("test_size",        len(X_test))

    mlflow.log_metrics(METRICS)
    mlflow.log_metric("auc_roc_validation", round(val_auc, 4))
    mlflow.log_metric("dataset_total_size", N)

    mlflow.set_tag("model_type",             "TF-IDF + LinearSVC (calibrated)")
    mlflow.set_tag("framework",              "scikit-learn")
    mlflow.set_tag("data_source",            "Textes medicaux synthetiques")
    mlflow.set_tag("data_language",          "francais")
    mlflow.set_tag("contains_personal_data", "non — textes synthetiques")
    mlflow.set_tag("protected_attributes",   "aucun")
    mlflow.set_tag("threshold_accuracy",     "0.88")
    mlflow.set_tag("threshold_f1",           "0.85")

    mlflow.set_tag("model.author",      "Octo Technology MLOps Tribe")
    mlflow.set_tag("model.team",        TEAM)
    mlflow.set_tag("model.environment", ENVIRONMENT)
    mlflow.set_tag("data.synthetic",    "true")
    mlflow.set_tag("pandera.status",    PANDERA_STATUS)
    mlflow.set_tag("ai_act.risk_level", "minimal")
    mlflow.set_tag("ai_act.annex_ref",  "N/A")
    mlflow.set_tag("ai_act.domain",     "Gestion documentaire medicale")
    mlflow.set_tag("mlflow.note.content", model_card_text)

    for art in [cr_path, fi_csv_path, fi_path, fairness_path, roc_path, pp_path, validation_report_path, model_card_path]:
        mlflow.log_artifact(str(art))
    if SCHEMA_YAML_EXPORTED:
        mlflow.log_artifact(str(schema_yaml_path))

    # Log pipeline (vectorizer + model) comme modele MLflow
    from sklearn.pipeline import Pipeline
    pipeline = Pipeline([("tfidf", vectorizer), ("clf", model)])
    signature     = infer_signature(X_train.tolist(), model.predict(X_train_sc))
    input_example = X_train.head(3).tolist()
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="custom_model",
        registered_model_name=MODEL_NAME,
        input_example={"document_type_classifier_input": input_example},
    )
    run_id = run.info.run_id

print(f"\nRun MLflow : {run_id}")
print(f"   Modele enregistre : {MODEL_NAME}")
print(f"   Accuracy : {METRICS['accuracy']}  |  F1 : {METRICS['f1_score']}  |  AUC : {METRICS['auc_roc']}")